In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
import os
import pickle

import sys
sys.path.append("/home/z5297792/ESP_zonodo")
from functions import doppio, out_core_param_fit

sys.path.append('/home/z5297792/UNSW-MRes/MRes/SEACOFS_dataset') 
from clim_functions import doppio_pipeliner, find_directional_radii, rotate_uv, transect_indexer, nearest_ij, interp_3d_to_reference_depths, local_eddy_arrays, add_reaches_50m_flag_fname


In [2]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)
lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))
def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
j_mid, i_mid = lon_rho.shape[1] // 2, lon_rho.shape[0] // 2
dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])
x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [3]:
df_data = pd.read_pickle('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/DOPPIO_SEACOFS_26yr_50m_vert_check/df_DOPPIO_SEACOFS_26yr_cleaned.pkl')


In [ ]:
# # add a flag to df_data 1=the edepth df gets to a depth of at least 50m, and 0 otherwise.
worker_id = 1  # change to 0, 1, 2, 3

save_file = (
    f'/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/DOPPIO_SEACOFS_26yr_50m_vert_check/'
    f'df_data_checked_part{worker_id}.pkl'
)

fname_chunks = np.array_split(np.sort(df_data.fname.dropna().unique()), 10)
fnames = fname_chunks[worker_id]
target_depths = np.abs(z_r[150,150,1:])

if os.path.exists(save_file):
    df_part = pd.read_pickle(save_file)
else:
    df_part = df_data[df_data.fname.isin(fnames)].copy()
    df_part['reaches_50m'] = pd.Series(
        pd.NA,
        index=df_part.index,
        dtype='boolean'
    )

for i, fname in enumerate(fnames, start=1):

    done = df_part.loc[
        df_part.fname.eq(fname),
        'reaches_50m'
    ].notna().all()

    if done:
        continue

    print(f'[{i}/{len(fnames)}] {os.path.basename(fname)}')

    df_part = add_reaches_50m_flag_fname(
        df_part,
        fname,
        X_grid, Y_grid,
        angle, z_r,
        target_depths
    )

    df_part.to_pickle(save_file)

    n_true = df_part.loc[
        df_part.fname.eq(fname),
        'reaches_50m'
    ].sum()

    print(f'    saved ({n_true} reached 50 m)')
    

[1/31] outer_avg_02391.nc
